# v5 - FINAL, complete and self-contained
## Haemoglobin and red cell indices from peripheral blood smears: a benchmark audit

**Self-contained.** Requires only the three input datasets (AneRBC, manifests, lab data).
Reads nothing from any other notebook's output.

### The mask-polarity bug, fixed here
`Binary_segmented` masks store **cells as DARK** (low values) on a bright background.
All earlier runs used `mask > 127`, which selected the **background**. Every
handcrafted feature in v1-v4 therefore described empty space between cells.
This notebook uses `mask < 127` throughout and re-runs everything affected.

Verified on slide 001_a: `> 127` gives 2 components; `< 127` gives 1,480 components
with 1,012 in the plausible cell-size range (the dataset paper states ~1,550 RBCs
per image).

### What this notebook establishes

| Part | Content | Time |
|---|---|---|
| A | Benchmark audit: label validity, tiling proof, record integrity | 3 min |
| B | Geometric measurement vs analyser indices (no model) | 15 min |
| C | 4-way ablation, **re-run with corrected masks** | 50 min |
| D | Binary anemia decision: WHO truth vs folder label | - |
| E | Abstention: does uncertainty track error? | - |
| F | Unique-CBC diagnostic (target noise) | 12 min |
| G | ResNet50 capacity check | 4 min |
| H | Lab OOD probe + feature-range diagnostic | 5 min |
| H2 | Lab batch 2: external slides **with analyser Hb** (inference only) | 1 min |

Part H2 needs `labdata_batch2_cbc.csv` and the `FY*_40X10_X.jpg` images in the lab
dataset, next to `labdata_annotations.csv`; if they are absent it skips itself.

**Total ~90 minutes.** Every run is checkpointed to `cache5/`, so an interruption
costs minutes rather than hours.


## 0 - Configuration


In [1]:
import os, sys, glob, json, time, random, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from scipy.stats import spearmanr, pearsonr, mannwhitneyu, ttest_rel
warnings.filterwarnings("ignore")

CFG = dict(
    SLIDE_PX=448,
    MASK_CELLS_ARE_DARK=True,   # <-- the fix; see the header
    N_FOLDS=5, SEEDS=[0,1,2],
    EPOCHS_FR=45, EPOCHS_FT=35,
    BATCH=32, LR_HEAD=1e-3, LR_BACK=1e-4, WD=1e-4,
    DROPOUT=0.3, MC_PASSES=30, HB_TOL=1.5,
    HB_CLAMP=(3.0,22.0), WHO_CUT=12.0,
    WORKERS=1,
    CELL_AREA_RANGE=(50,20000),
)
RUN = dict(AUDIT=True, GEOMETRY=True, ABLATION=True,
           UNIQUE_CBC=True, RESNET50=True, LAB=True)

WORK=Path("/kaggle/working")
for d in ("results5","figures5","cache5"): (WORK/d).mkdir(parents=True,exist_ok=True)
DEV="cuda" if torch.cuda.is_available() else "cpu"
print("device:",DEV, torch.cuda.get_device_name(0) if DEV=="cuda" else "")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def banner(t): print("\n"+"="*74); print(t); print("="*74)


device: cuda Tesla T4


## 1 - Locate inputs and build the cohort


In [2]:
INPUT=Path("/kaggle/input")
ANERBC=next((p.parent for p in INPUT.rglob("AneRBC-I") if p.is_dir()), None)
MAN   =next((p.parent for p in INPUT.rglob("anerbc_patients.csv")), None)
LABDIR=next((p.parent for p in INPUT.rglob("labdata_annotations.csv")), None)
assert ANERBC and MAN and LABDIR, (ANERBC, MAN, LABDIR)
print("AneRBC  :",ANERBC); print("manifest:",MAN); print("lab     :",LABDIR)

def d_img(s,g): return ANERBC/s/f"{g}_individuals"/"Original_images"
def d_msk(s,g): return ANERBC/s/f"{g}_individuals"/"Binary_segmented"
def sp(pid,kind="img"):
    g="Anemic" if pid.endswith("_a") else "Healthy"
    return str((d_img("AneRBC-I",g) if kind=="img" else d_msk("AneRBC-I",g))/f"{pid}.png")

def read_mask(path, size=None):
    """Cells are DARK in Binary_segmented. Returns a boolean CELL mask."""
    im=Image.open(path).convert("L")
    if size: im=im.resize(size, Image.NEAREST)
    a=np.asarray(im)
    return (a<127) if CFG["MASK_CELLS_ARE_DARK"] else (a>127)

patients=pd.read_csv(MAN/"anerbc_patients.csv")
clean=patients[(~patients.unusable)&patients.HGB.notna()].copy().reset_index(drop=True)
clean["slide"]=clean.patient_id.map(lambda p: sp(p,"img"))
clean["smask"]=clean.patient_id.map(lambda p: sp(p,"msk"))
clean=clean[clean.slide.map(os.path.exists)].reset_index(drop=True)
clean["anemic_who"]=(clean.HGB<CFG["WHO_CUT"]).astype(int)
if "fingerprint" in clean:
    vc=clean.fingerprint.value_counts(); clean["cbc_unique"]=clean.fingerprint.map(vc).eq(1)
else: clean["cbc_unique"]=True
BASE_MAE=(clean.HGB-clean.HGB.median()).abs().mean()

print(f"\npatients {len(clean)} | unique CBC {int(clean.cbc_unique.sum())}")
print(f"HGB mean {clean.HGB.mean():.2f} sd {clean.HGB.std():.2f} "
      f"[{clean.HGB.min():.1f},{clean.HGB.max():.1f}]")
print(f"baseline MAE (median predictor) {BASE_MAE:.3f} g/dL")
for c in ("MCV","RDW","MCH","MCHC","RBC","HCT"):
    if c in clean: print(f"  {c:5s} present for {clean[c].notna().sum():4d}")


AneRBC  : /kaggle/input/datasets/nehalshivane/anerbc-dataset/AneRBC dataset a benchmark dataset for computer-aided anemia diagnosis using RBC images. httpsdoi.org10.1093databasebaae120/AneRBC_dataset
manifest: /kaggle/input/datasets/nehalshivane/anerbc-manifests
lab     : /kaggle/input/datasets/nehalshivane/lab-complete

patients 917 | unique CBC 363
HGB mean 10.77 sd 3.13 [3.5,20.6]
baseline MAE (median predictor) 2.549 g/dL
  MCV   present for  917
  RDW   present for  578
  MCH   present for  917
  MCHC  present for  909
  RBC   present for  916
  HCT   present for  917


In [3]:
# VERIFY the mask polarity before anything depends on it
from skimage.measure import label as sklabel, regionprops
pid=clean.patient_id.iloc[0]
a=np.asarray(Image.open(sp(pid,"msk")).convert("L"))
banner(f"MASK POLARITY CHECK ({pid})")
for name,mk in (("a > 127 (background)", a>127), ("a < 127 (cells)", a<127)):
    ar=np.array([p.area for p in regionprops(sklabel(mk))])
    lo,hi=CFG["CELL_AREA_RANGE"]
    good=int(((ar>=lo)&(ar<=hi)).sum()) if len(ar) else 0
    print(f"  {name:22s} fg={mk.mean():.3f}  components={len(ar):5d}  in-range={good:5d}")
print("  dataset paper states ~1550 RBCs per AneRBC-I image")
print(f"  -> using MASK_CELLS_ARE_DARK = {CFG['MASK_CELLS_ARE_DARK']}")



MASK POLARITY CHECK (001_a)
  a > 127 (background)   fg=0.653  components=    2  in-range=    1
  a < 127 (cells)        fg=0.347  components= 1480  in-range= 1012
  dataset paper states ~1550 RBCs per AneRBC-I image
  -> using MASK_CELLS_ARE_DARK = True


---
# PART A - Benchmark audit


In [4]:
audit={}
if RUN["AUDIT"]:
    pos=clean.loc[clean.label==1,"HGB"].values; neg=clean.loc[clean.label==0,"HGB"].values
    r=pd.Series(-np.concatenate([pos,neg])).rank().values
    audit["label_vs_hgb_auc"]=round(float(
        (r[:len(pos)].sum()-len(pos)*(len(pos)+1)/2)/(len(pos)*len(neg))),3)
    for lab,n in ((0,"healthy"),(1,"anemic")):
        audit[f"who_anemic_pct_{n}_folder"]=round(
            100*float((clean.loc[clean.label==lab,"HGB"]<CFG["WHO_CUT"]).mean()),1)
    audit["label_vs_who_agreement_pct"]=round(100*float((clean.label==clean.anemic_who).mean()),1)
    audit["label_who_disagreement_pct"]=round(100-audit["label_vs_who_agreement_pct"],1)
    if "morph_abnormal" in patients:
        audit["label_vs_morphology_pct"]=round(
            100*float(((patients.label==1)==patients.morph_abnormal).mean()),1)
    if "fingerprint" in patients:
        audit["cbc_files"]=int(len(patients))
        audit["cbc_distinct"]=int(patients.fingerprint.nunique())
        audit["cbc_duplicate_pct"]=round(100*(1-patients.fingerprint.nunique()/len(patients)),1)
    audit["unusable_records"]=int(patients.unusable.sum())
    audit["clean_patients"]=int(len(clean))
    banner("PART A - AUDIT")
    for k,v in audit.items(): print(f"  {k:32s} {v}")
    pd.Series(audit).to_csv(WORK/"results5"/"table1_audit.csv")



PART A - AUDIT
  label_vs_hgb_auc                 0.635
  who_anemic_pct_healthy_folder    50.1
  who_anemic_pct_anemic_folder     72.5
  label_vs_who_agreement_pct       61.4
  label_who_disagreement_pct       38.6
  label_vs_morphology_pct          88.1
  cbc_files                        1000
  cbc_distinct                     641
  cbc_duplicate_pct                35.9
  unusable_records                 82
  clean_patients                   917


In [5]:
if RUN["AUDIT"]:
    def stitch(pid="001_a"):
        g="Anemic" if pid.endswith("_a") else "Healthy"; L=pid[-1]; n=int(pid.split("_")[0])
        pf=d_img("AneRBC-I",g)/f"{pid}.png"
        ts=[d_img("AneRBC-II",g)/f"{n:04d}_{i:02d}_{L}.png" for i in range(1,13)]
        if not pf.exists() or not all(t.exists() for t in ts):
            print("files missing"); return None
        par=Image.open(pf).convert("RGB"); pw,ph=par.size
        tl=[Image.open(t).convert("RGB") for t in ts]; tw,th=tl[0].size
        cv=Image.new("RGB",(pw,ph))
        for i,t in enumerate(tl): cv.paste(t,((i%4)*tw,(i//4)*th))
        d=float(np.abs(np.asarray(cv,np.int16)-np.asarray(par,np.int16)).mean())
        f,ax=plt.subplots(1,2,figsize=(13,5))
        ax[0].imshow(par); ax[0].set_title(f"AneRBC-I original ({pid})")
        ax[1].imshow(cv);  ax[1].set_title(f"12 AneRBC-II tiles restitched 4x3\n"
                                           f"mean |pixel difference| = {d:.4f}")
        for a_ in ax: a_.axis("off")
        plt.tight_layout(); plt.savefig(WORK/"figures5"/"fig1_stitch.png",dpi=160); plt.show()
        print(f"mean absolute pixel difference = {d:.4f}")
        print(f"P(test tile has no same-slide sibling in an 80/10/10 split) = {0.2**11:.2e}")
        return d
    audit["stitch_pixel_diff"]=stitch()

    f,ax=plt.subplots(1,2,figsize=(12,4))
    b=np.linspace(clean.HGB.min(),clean.HGB.max(),40)
    ax[0].hist(clean.loc[clean.label==0,"HGB"],bins=b,alpha=.6,
               label="'Healthy' folder",color="seagreen")
    ax[0].hist(clean.loc[clean.label==1,"HGB"],bins=b,alpha=.6,
               label="'Anemic' folder",color="indianred")
    ax[0].axvline(CFG["WHO_CUT"],ls="--",c="k",lw=1.2,label="WHO threshold")
    ax[0].set_xlabel("measured haemoglobin (g/dL)"); ax[0].set_ylabel("patients")
    ax[0].set_title(f"folder label vs measured HGB   AUC = {audit['label_vs_hgb_auc']}")
    ax[0].legend(fontsize=8)
    ax[1].boxplot([clean.loc[clean.label==0,"HGB"],clean.loc[clean.label==1,"HGB"]],
                  labels=["Healthy folder","Anemic folder"])
    ax[1].axhline(CFG["WHO_CUT"],ls="--",c="k",lw=1.2); ax[1].set_ylabel("HGB (g/dL)")
    ax[1].set_title("overlapping distributions")
    plt.tight_layout(); plt.savefig(WORK/"figures5"/"fig2_label_vs_hgb.png",dpi=160); plt.show()


[output removed: shows per-slide external laboratory data, which is not shared to protect patient privacy]


---
# PART B - Geometric measurement vs analyser indices

No model, no training. For every patient, every segmented cell is measured at
native 1224x960 and aggregated, then correlated against the Sysmex values.

Also tested: the **area-plus-central-pallor** correction described in
US Patent 4,453,266, which claims that projected area alone is insufficient for
MCV and requires a pallor term to account for the biconcave depth lost when
cells are spread and dried.

Prior work: Zhao et al., Front Med 2022 (n=182) reported no significant
correlation between smear RBC area and MCV, MCH, MCHC or RDW.


In [6]:
if RUN["GEOMETRY"]:
    LO,HI=CFG["CELL_AREA_RANGE"]

    def cell_stats(pid, max_cells=3000):
        mp,ip=sp(pid,"msk"), sp(pid,"img")
        if not os.path.exists(mp): return None
        m=read_mask(mp)
        g=np.asarray(Image.open(ip).convert("L")).astype(float)
        props=[p for p in regionprops(sklabel(m), intensity_image=g)
               if LO<=p.area<=HI]
        if len(props)<20: return None
        props=props[:max_cells]
        areas,ecc,sol,rimv,palf,pald=[],[],[],[],[],[]
        for p in props:
            v=p.intensity_image.astype(float)[p.image]
            if v.size<20: continue
            thr=np.percentile(v,60); pal=v>thr        # pale centre vs stained rim
            areas.append(p.area); ecc.append(p.eccentricity); sol.append(p.solidity)
            palf.append(float(pal.mean()))
            pald.append(float(v[pal].mean()-v[~pal].mean()))
            rimv.append(float(v[~pal].mean()))
        if len(areas)<20: return None
        a=np.array(areas,float)
        return dict(patient_id=pid, n_cells=len(a),
            area_mean=a.mean(), area_median=float(np.median(a)), area_sd=a.std(),
            area_cv=a.std()/a.mean(),
            diam_mean=float(np.sqrt(4*a.mean()/np.pi)),
            diam_cv=float(np.sqrt(a).std()/np.sqrt(a).mean()),
            ecc_mean=float(np.mean(ecc)), sol_mean=float(np.mean(sol)),
            pal_frac=float(np.mean(palf)), pal_depth=float(np.mean(pald)),
            rim_int=float(np.mean(rimv)),
            vol_proxy=float(a.mean()*(1-np.mean(palf))),
            vol_proxy2=float(a.mean()*np.mean(pald)))

    ck=WORK/"cache5"/f"cellstats_{len(clean)}.csv"
    if ck.exists(): G=pd.read_csv(ck); print("cache:",len(G))
    else:
        rows,t0=[],time.time()
        for i,pid in enumerate(clean.patient_id):
            s=cell_stats(pid)
            if s: rows.append(s)
            if i%100==0: print(f"  {i}/{len(clean)} kept={len(rows)} "
                               f"{time.time()-t0:.0f}s",end="\r")
        G=pd.DataFrame(rows); G.to_csv(ck,index=False)
        print(f"\nmeasured {len(G)}/{len(clean)} slides in {time.time()-t0:.0f}s")
    print(f"median cells per slide: {G.n_cells.median():.0f}  "
          f"(dataset paper states ~1550 RBCs per image)")
    ana=[c for c in ("HGB","MCV","RDW","MCH","MCHC","RBC","HCT","label","anemic_who")
         if c in clean.columns]
    G=G.drop(columns=[c for c in ana if c in G.columns],errors="ignore")
    G=G.merge(clean[["patient_id"]+ana],on="patient_id",how="left")


  900/917 kept=891 1025s
measured 907/917 slides in 1049s
median cells per slide: 1038  (dataset paper states ~1550 RBCs per image)


In [7]:
if RUN["GEOMETRY"]:
    pairs=[("area_mean","MCV"),("area_median","MCV"),("diam_mean","MCV"),
           ("vol_proxy","MCV"),("vol_proxy2","MCV"),
           ("area_cv","RDW"),("diam_cv","RDW"),("area_sd","RDW"),
           ("pal_frac","MCHC"),("pal_depth","MCHC"),("rim_int","MCHC"),
           ("area_mean","MCH"),("vol_proxy2","MCH"),
           ("n_cells","RBC"),("n_cells","HCT"),("n_cells","HGB"),
           ("rim_int","HGB")]
    rows=[]
    for f,t in pairs:
        if f not in G.columns or t not in G.columns: continue
        d=G[[f,t]].dropna()
        if len(d)<30: continue
        pr,pp=pearsonr(d[f],d[t]); sr,_=spearmanr(d[f],d[t])
        rows.append(dict(measured=f,analyser=t,n=len(d),pearson_r=round(pr,3),
                         spearman=round(sr,3),r2=round(pr**2,3),p=f"{pp:.1e}"))
    geo=pd.DataFrame(rows).sort_values("r2",ascending=False)
    banner("PART B - GEOMETRY vs ANALYSER (single predictors)")
    print(geo.to_string(index=False))
    geo.to_csv(WORK/"results5"/"table2_geometry.csv",index=False)

    # multivariate, cross-validated: does the patented pallor correction help?
    from sklearn.linear_model import LinearRegression
    from sklearn.model_selection import KFold
    from sklearn.metrics import r2_score
    models={"area only":["area_mean"],
            "area + pallor frac":["area_mean","pal_frac"],
            "area + pallor depth":["area_mean","pal_depth"],
            "patent-style (area+pallor+rim)":["area_mean","pal_frac","rim_int"],
            "all geometry":["area_mean","area_cv","pal_frac","pal_depth",
                            "rim_int","vol_proxy","vol_proxy2","ecc_mean","sol_mean"]}
    mrows=[]
    for tgt in ("MCV","MCH","MCHC","RDW"):
        if tgt not in G.columns: continue
        for name,fe in models.items():
            fe=[f for f in fe if f in G.columns]
            d=G[fe+[tgt]].dropna()
            if len(d)<50: continue
            X,y=d[fe].values,d[tgt].values
            pr=np.zeros_like(y,dtype=float)
            for tr,te in KFold(5,shuffle=True,random_state=0).split(X):
                pr[te]=LinearRegression().fit(X[tr],y[tr]).predict(X[te])
            mrows.append(dict(target=tgt,model=name,n=len(d),
                              cv_r2=round(r2_score(y,pr),3),
                              cv_r=round(float(np.corrcoef(pr,y)[0,1]),3)))
    mv=pd.DataFrame(mrows)
    banner("PART B - MULTIVARIATE (5-fold CV): does the pallor correction help?")
    print(mv.to_string(index=False))
    mv.to_csv(WORK/"results5"/"table3_pallor_correction.csv",index=False)
    print("\nREFERENCE POINTS")
    print("  Zhao et al. 2022 (n=182): no significant area-MCV correlation")
    print("  human visual anisocytosis vs analyser RDW: Kendall tau 0.37-0.51")
    print("  Yang et al. 2026 CNN: MCV PCC 0.985, MCHC PCC 0.688")
    print("  US Patent 4,453,266: claims area + central pallor is required for MCV")



PART B - GEOMETRY vs ANALYSER (single predictors)
   measured analyser   n  pearson_r  spearman    r2       p
 vol_proxy2      MCH 907      0.195     0.153 0.038 3.3e-09
 vol_proxy2      MCV 907      0.192     0.139 0.037 5.6e-09
area_median      MCV 907      0.174     0.195 0.030 1.3e-07
  diam_mean      MCV 907      0.164     0.128 0.027 7.1e-07
  area_mean      MCV 907      0.162     0.128 0.026 9.2e-07
  vol_proxy      MCV 907      0.160     0.129 0.026 1.3e-06
  area_mean      MCH 907      0.153     0.124 0.023 3.8e-06
  pal_depth     MCHC 899      0.135     0.102 0.018 4.8e-05
    n_cells      RBC 906      0.108     0.077 0.012 1.1e-03
   pal_frac     MCHC 899      0.090     0.116 0.008 6.7e-03
    rim_int      HGB 907      0.049     0.056 0.002 1.4e-01
    n_cells      HCT 907      0.044     0.044 0.002 1.8e-01
    rim_int     MCHC 899     -0.022    -0.025 0.001 5.0e-01
    n_cells      HGB 907      0.027     0.035 0.001 4.1e-01
    area_sd      RDW 572     -0.010    -0.016 0.0

In [8]:
if RUN["GEOMETRY"]:
    pp=[(f,t) for f,t in (("area_mean","MCV"),("area_cv","RDW"),("pal_depth","MCHC"))
        if f in G.columns and t in G.columns and len(G[[f,t]].dropna())>30]
    if pp:
        fig,ax=plt.subplots(1,len(pp),figsize=(5.5*len(pp),4.5))
        if len(pp)==1: ax=[ax]
        for a_,(f,t) in zip(ax,pp):
            d=G[[f,t]].dropna(); rr,_=pearsonr(d[f],d[t])
            a_.scatter(d[f],d[t],s=10,alpha=.35,c="steelblue")
            z=np.polyfit(d[f],d[t],1); xs=np.linspace(d[f].min(),d[f].max(),50)
            a_.plot(xs,np.polyval(z,xs),"r--",lw=1.2)
            a_.set_xlabel(f"measured {f}"); a_.set_ylabel(f"analyser {t}")
            a_.set_title(f"r = {rr:.3f}   r2 = {rr**2:.3f}   n = {len(d)}")
        plt.tight_layout(); plt.savefig(WORK/"figures5"/"fig3_geometry.png",dpi=160); plt.show()


[output removed: shows per-slide external laboratory data, which is not shared to protect patient privacy]


---
# Data loading and model (corrected masks)


In [9]:
SP=CFG["SLIDE_PX"]
cf=WORK/"cache5"/f"slides_{SP}_{len(clean)}.npz"
if cf.exists():
    z=np.load(cf); SLIDES,SMASKS=z["s"],z["m"]; print("cache:",SLIDES.shape)
else:
    SLIDES=np.zeros((len(clean),SP,SP,3),np.uint8); SMASKS=np.zeros((len(clean),SP,SP),bool)
    t0=time.time()
    for i,r in clean.iterrows():
        SLIDES[i]=np.asarray(Image.open(r.slide).convert("RGB").resize((SP,SP),Image.LANCZOS))
        if os.path.exists(r.smask):
            SMASKS[i]=read_mask(r.smask,(SP,SP))        # corrected polarity
        if i%200==0: print(f"  {i}/{len(clean)} {time.time()-t0:.0f}s",end="\r")
    np.savez_compressed(cf,s=SLIDES,m=SMASKS); print(f"\n{time.time()-t0:.0f}s")
print(f"slides {SLIDES.shape} | mask CELL fraction {SMASKS.mean():.3f}  "
      f"(should be ~0.3-0.4, not ~0.65)")


  800/917 66s
114s
slides (917, 448, 448, 3) | mask CELL fraction 0.369  (should be ~0.3-0.4, not ~0.65)


In [10]:
from skimage.feature import graycomatrix, graycoprops
from skimage.filters import threshold_otsu
FEATS=["area_mean","area_std","perim_mean","circ_mean","ecc_mean","solidity_mean",
       "glcm_contrast","glcm_correlation","glcm_energy","glcm_homogeneity",
       "mean_R","mean_G","mean_B","pallor_ratio"]
LO,HI=CFG["CELL_AREA_RANGE"]

def extract_14d(rgb, mask=None):
    gray=np.asarray(Image.fromarray(rgb).convert("L"))
    if mask is None or mask.sum()<50:
        try: mask=gray<threshold_otsu(gray)          # cells are dark
        except Exception: mask=gray<gray.mean()
    props=[p for p in regionprops(sklabel(mask.astype(bool))) if 20<=p.area<=HI]
    if props:
        a=np.array([p.area for p in props],float)
        pr=np.array([p.perimeter for p in props],float)
        cr=4*np.pi*a/(pr**2+1e-6)
        ec=np.array([p.eccentricity for p in props],float)
        so=np.array([p.solidity for p in props],float)
        morph=[a.mean(),a.std(),pr.mean(),cr.mean(),ec.mean(),so.mean()]
    else: morph=[0.]*6
    q=(np.asarray(Image.fromarray(gray).resize((160,160)))//8).astype(np.uint8)
    gl=graycomatrix(q,[1],[0],levels=32,symmetric=True,normed=True)
    tex=[float(graycoprops(gl,p)[0,0]) for p in
         ("contrast","correlation","energy","homogeneity")]
    col=[float(rgb[...,c].mean()) for c in range(3)]
    h,w=gray.shape; ctr=gray[h//2-h//8:h//2+h//8, w//2-w//8:w//2+w//8]
    return np.array(morph+tex+col+[float((ctr>gray.mean()).mean())],np.float32)

def dihedral(a,k):
    o=np.rot90(a,k%4)
    return np.ascontiguousarray(o[:,::-1] if k>=4 else o)

hf=WORK/"cache5"/f"hand8_{SP}_{len(clean)}.npy"
if hf.exists(): HAND8=np.load(hf); print("cache:",HAND8.shape)
else:
    HAND8=np.zeros((len(clean),8,14),np.float32); t0=time.time()
    for i in range(len(clean)):
        for k in range(8): HAND8[i,k]=extract_14d(dihedral(SLIDES[i],k),dihedral(SMASKS[i],k))
        if i%50==0: print(f"  {i}/{len(clean)} {time.time()-t0:.0f}s",end="\r")
    np.save(hf,HAND8); print(f"\n{time.time()-t0:.0f}s")
print("sanity - mean area_mean across patients:", round(float(HAND8[:,0,0].mean()),1),
      "px  (background-derived values were far larger)")


  900/917 3410s
3496s
sanity - mean area_mean across patients: 107.3 px  (background-derived values were far larger)


In [11]:
IMNET_M=torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
IMNET_S=torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)

class SlideDS(Dataset):
    def __init__(self,idx,y,train,hand):
        self.idx=np.asarray(idx); self.y=y; self.train=train; self.hand=hand
    def __len__(self): return len(self.idx)
    def __getitem__(self,j):
        i=self.idx[j]; k=np.random.randint(8) if self.train else 0
        img=dihedral(SLIDES[i],k).astype(np.float32)/255.
        return (torch.from_numpy(img.transpose(2,0,1).copy()),
                torch.from_numpy(self.hand[i,k]),
                torch.tensor(self.y[j],dtype=torch.float32))

class HybridNet(nn.Module):
    """Adaptive softmax fusion of a deep stream and a handcrafted stream.
    This mechanism is established prior work (e.g. arXiv:2308.13997); it is used
    here as a method, not claimed as a contribution."""
    def __init__(self,freeze=True,use_hand=True,p=0.3,arch="resnet18"):
        super().__init__()
        if arch=="resnet18": self.bb=torchvision.models.resnet18(weights="IMAGENET1K_V1"); dc=512
        else: self.bb=torchvision.models.resnet50(weights="IMAGENET1K_V2"); dc=2048
        self.bb.fc=nn.Identity(); self.freeze=freeze; self.use_hand=use_hand
        if freeze:
            for q in self.bb.parameters(): q.requires_grad=False
        self.score=nn.Sequential(nn.Linear(dc+14,64),nn.ReLU(),nn.Linear(64,2))
        self.pc=nn.Linear(dc,256); self.ph=nn.Linear(14,256)
        self.head=nn.Sequential(nn.Dropout(p),nn.Linear(256,128),nn.ReLU(),
                                nn.Dropout(p),nn.Linear(128,1))
    def forward(self,img,hand,return_alpha=False):
        img=(img-IMNET_M.to(img.device))/IMNET_S.to(img.device)
        if self.freeze:
            self.bb.eval()
            with torch.no_grad(): xc=self.bb(img)
        else: xc=self.bb(img)
        xh=hand if self.use_hand else torch.zeros_like(hand)
        a=torch.softmax(self.score(torch.cat([xc,xh],1)),1)
        f=a[:,:1]*F.relu(self.pc(xc))+a[:,1:]*F.relu(self.ph(xh))
        y=self.head(f).squeeze(1)
        return (y,a) if return_alpha else y

def pgroups(m):
    back=[p for p in m.bb.parameters() if p.requires_grad]
    rest=[p for n,p in m.named_parameters() if not n.startswith("bb.") and p.requires_grad]
    g=[dict(params=rest,lr=CFG["LR_HEAD"])]
    if back: g.append(dict(params=back,lr=CFG["LR_BACK"]))
    return g

from sklearn.model_selection import StratifiedKFold
clean["hgb_bin"]=pd.qcut(clean.HGB,5,labels=False,duplicates="drop")
FOLDS=list(StratifiedKFold(CFG["N_FOLDS"],shuffle=True,random_state=0).split(clean,clean.hgb_bin))
for k,(tr,te) in enumerate(FOLDS): print(f"fold {k}: train {len(tr)} | test {len(te)}")

def train_eval(fold,seed,freeze,use_hand,arch="resnet18",train_subset=None,tag=""):
    ck=WORK/"cache5"/f"r_{tag}_{arch}_{int(freeze)}{int(use_hand)}_f{fold}_s{seed}.csv"
    if ck.exists(): return pd.read_csv(ck)
    set_seed(seed)
    tr,te=FOLDS[fold]
    if train_subset is not None: tr=np.array([i for i in tr if train_subset[i]])
    y=clean.HGB.values.astype(np.float32)
    mu_h=HAND8[tr].reshape(-1,14).mean(0); sd_h=HAND8[tr].reshape(-1,14).std(0)+1e-6
    mu_y,sd_y=y[tr].mean(),y[tr].std()
    HN=((HAND8-mu_h)/sd_h).astype(np.float32)
    dtr=DataLoader(SlideDS(tr,(y[tr]-mu_y)/sd_y,True,HN),batch_size=CFG["BATCH"],
                   shuffle=True,num_workers=CFG["WORKERS"],drop_last=True)
    dte=DataLoader(SlideDS(te,(y[te]-mu_y)/sd_y,False,HN),batch_size=CFG["BATCH"],
                   shuffle=False,num_workers=CFG["WORKERS"])
    m=HybridNet(freeze,use_hand,CFG["DROPOUT"],arch).to(DEV)
    ep=CFG["EPOCHS_FR"] if freeze else CFG["EPOCHS_FT"]
    opt=torch.optim.AdamW(pgroups(m),weight_decay=CFG["WD"])
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=ep)
    sc=torch.amp.GradScaler("cuda",enabled=(DEV=="cuda"))
    for e in range(ep):
        m.train()
        for img,hd,yb in dtr:
            img,hd,yb=img.to(DEV,non_blocking=True),hd.to(DEV),yb.to(DEV)
            with torch.amp.autocast("cuda",enabled=(DEV=="cuda")):
                loss=F.huber_loss(m(img,hd),yb,delta=1.0)
            opt.zero_grad(); sc.scale(loss).backward(); sc.step(opt); sc.update()
        sch.step()
    preds,alphas,mcs=[],[],[]
    m.eval()
    with torch.no_grad():
        for img,hd,_ in dte:
            p,a=m(img.to(DEV),hd.to(DEV),return_alpha=True)
            preds.append(p.cpu().numpy()); alphas.append(a.cpu().numpy())
    for mod in m.modules():
        if isinstance(mod,nn.Dropout): mod.train()
    with torch.no_grad():
        for img,hd,_ in dte:
            img,hd=img.to(DEV),hd.to(DEV)
            mcs.append(np.stack([m(img,hd).cpu().numpy() for _ in range(CFG["MC_PASSES"])],1))
    lo,hi=CFG["HB_CLAMP"]
    o=clean.iloc[te][["patient_id","HGB","label","anemic_who","cbc_unique"]].copy()
    o["pred"]=np.clip(np.concatenate(preds)*sd_y+mu_y,lo,hi)
    mc=np.concatenate(mcs)*sd_y+mu_y
    o["mc_mean"]=np.clip(mc.mean(1),lo,hi); o["mc_std"]=mc.std(1)
    o["alpha_hand"]=np.concatenate(alphas)[:,1]
    o["fold"],o["seed"],o["n_train"]=fold,seed,len(tr)
    o.to_csv(ck,index=False)
    del m; torch.cuda.empty_cache()
    return o

def summarise(df,key="exp"):
    rows=[]
    for k,g in df.groupby(key):
        per=g.groupby(["fold","seed"]).apply(lambda d: pd.Series({
            "MAE":np.abs(d.pred-d.HGB).mean(),
            "RMSE":np.sqrt(((d.pred-d.HGB)**2).mean()),
            "R2":1-((d.pred-d.HGB)**2).sum()/((d.HGB-d.HGB.mean())**2).sum(),
            "PCC":np.corrcoef(d.pred,d.HGB)[0,1]}))
        rows.append({key:k,
            "MAE":f"{per.MAE.mean():.3f} ± {per.MAE.std():.3f}",
            "RMSE":f"{per.RMSE.mean():.3f} ± {per.RMSE.std():.3f}",
            "R2":f"{per.R2.mean():.3f} ± {per.R2.std():.3f}",
            "PCC":f"{per.PCC.mean():.3f} ± {per.PCC.std():.3f}","_m":per.MAE.mean()})
    return pd.DataFrame(rows).sort_values("_m")


fold 0: train 733 | test 184
fold 1: train 733 | test 184
fold 2: train 734 | test 183
fold 3: train 734 | test 183
fold 4: train 734 | test 183


---
# PART C - 4-way ablation, corrected masks (~50 min)


In [12]:
EXPS=[("A_frozen_cnn",True,False),("B_frozen_hybrid",True,True),
      ("C_ft_cnn",False,False),("D_ft_hybrid",False,True)]
if RUN["ABLATION"]:
    R,t0=[],time.time()
    for name,fz,uh in EXPS:
        for f in range(CFG["N_FOLDS"]):
            for s in CFG["SEEDS"]:
                o=train_eval(f,s,fz,uh,tag="abl"); o["exp"]=name; R.append(o)
                print(f"{name:16s} f{f} s{s}  MAE={np.abs(o.pred-o.HGB).mean():.3f}"
                      f"  [{time.time()-t0:.0f}s]")
    res=pd.concat(R,ignore_index=True)
    res.to_csv(WORK/"results5"/"predictions_ablation.csv",index=False)
    print(f"\ntotal {time.time()-t0:.0f}s")

tab=summarise(res)
banner("PART C - 4-WAY ABLATION (corrected masks)")
print(tab.drop(columns="_m").to_string(index=False))
print(f"\nbaseline (median predictor) MAE = {BASE_MAE:.3f} g/dL")
tab.drop(columns="_m").to_csv(WORK/"results5"/"table4_ablation.csv",index=False)
BEST=tab.iloc[0].exp; print("best:",BEST)

def paired(e1,e2):
    a=res[res.exp==e1].groupby(["fold","seed"]).apply(lambda d: np.abs(d.pred-d.HGB).mean())
    b=res[res.exp==e2].groupby(["fold","seed"]).apply(lambda d: np.abs(d.pred-d.HGB).mean())
    j=pd.concat([a.rename("a"),b.rename("b")],axis=1).dropna()
    t,p=ttest_rel(j.a,j.b)
    return dict(comparison=f"{e1} vs {e2}",n_pairs=len(j),
                mean_diff=round(float((j.a-j.b).mean()),4),
                favours=e2 if (j.a-j.b).mean()>0 else e1,
                wins=f"{int((j.a>j.b).sum())}/{len(j)}",t=round(float(t),2),p=f"{p:.2e}")
pairs_tab=pd.DataFrame([paired("A_frozen_cnn","B_frozen_hybrid"),
                        paired("C_ft_cnn","D_ft_hybrid"),
                        paired("A_frozen_cnn","C_ft_cnn"),
                        paired("B_frozen_hybrid","D_ft_hybrid")])
banner("PAIRED COMPARISONS (matched on fold and seed)")
print(pairs_tab.to_string(index=False))
pairs_tab.to_csv(WORK/"results5"/"table5_paired.csv",index=False)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 242MB/s]


A_frozen_cnn     f0 s0  MAE=2.238  [215s]
A_frozen_cnn     f0 s1  MAE=2.319  [394s]
A_frozen_cnn     f0 s2  MAE=2.235  [572s]
A_frozen_cnn     f1 s0  MAE=2.204  [751s]
A_frozen_cnn     f1 s1  MAE=2.322  [929s]
A_frozen_cnn     f1 s2  MAE=2.193  [1108s]
A_frozen_cnn     f2 s0  MAE=2.298  [1287s]
A_frozen_cnn     f2 s1  MAE=2.293  [1465s]
A_frozen_cnn     f2 s2  MAE=2.283  [1643s]
A_frozen_cnn     f3 s0  MAE=2.500  [1823s]
A_frozen_cnn     f3 s1  MAE=2.511  [2001s]
A_frozen_cnn     f3 s2  MAE=2.486  [2180s]
A_frozen_cnn     f4 s0  MAE=2.273  [2358s]
A_frozen_cnn     f4 s1  MAE=2.369  [2536s]
A_frozen_cnn     f4 s2  MAE=2.261  [2716s]
B_frozen_hybrid  f0 s0  MAE=2.126  [2894s]
B_frozen_hybrid  f0 s1  MAE=2.101  [3073s]
B_frozen_hybrid  f0 s2  MAE=2.141  [3251s]
B_frozen_hybrid  f1 s0  MAE=2.102  [3430s]
B_frozen_hybrid  f1 s1  MAE=2.115  [3609s]
B_frozen_hybrid  f1 s2  MAE=2.107  [3787s]
B_frozen_hybrid  f2 s0  MAE=2.084  [3966s]
B_frozen_hybrid  f2 s1  MAE=2.092  [4145s]
B_frozen_hybrid 

---
# PART D - Binary anemia decision


In [13]:
from sklearn.metrics import roc_auc_score, confusion_matrix
b=res[res.exp==BEST]
rows=[]
for truth,name in ((b.anemic_who,"WHO ground truth (HGB < 12)"),
                   (b.label,"AneRBC folder label")):
    pb=(b.pred<CFG["WHO_CUT"]).astype(int)
    tn,fp,fn,tp=confusion_matrix(truth,pb).ravel()
    rows.append(dict(scored_against=name,accuracy=round((tp+tn)/len(b),3),
                     sensitivity=round(tp/(tp+fn+1e-9),3),
                     specificity=round(tn/(tn+fp+1e-9),3),
                     AUC=round(roc_auc_score(truth,-b.pred),3)))
binary=pd.DataFrame(rows)
banner("PART D - BINARY ANEMIA DECISION (same model, two reference labels)")
print(binary.to_string(index=False))
print("\nThe gap between the rows is the measurable cost of the label error.")
binary.to_csv(WORK/"results5"/"table6_binary.csv",index=False)

def band(h): return "severe" if h<8 else "moderate" if h<11 else "mild" if h<12 else "none"
b2=b.copy(); b2["t"]=b2.HGB.map(band); b2["p"]=b2.pred.map(band)
print(f"\nWHO severity band exact accuracy: {100*(b2.t==b2.p).mean():.1f}%")
print(pd.crosstab(b2.t,b2.p).to_string())



PART D - BINARY ANEMIA DECISION (same model, two reference labels)
             scored_against  accuracy  sensitivity  specificity   AUC
WHO ground truth (HGB < 12)     0.741        0.863        0.546 0.765
        AneRBC folder label     0.621        0.821        0.413 0.670

The gap between the rows is the measurable cost of the label error.

WHO severity band exact accuracy: 48.5%
p         mild  moderate  none  severe
t                                     
mild        34       174    80       0
moderate    94       662    90      39
none       158       315   578       8
severe      59       338    61      61


---
# PART E - Abstention: does uncertainty track error?


In [14]:
one=b[b.seed==CFG["SEEDS"][0]]
err=np.abs(one.pred-one.HGB); rho,pv=spearmanr(one.mc_std,err)
banner("PART E - ABSTENTION")
print(f"uncertainty-error Spearman rho = {rho:.3f}  (p = {pv:.2e}, n = {len(one)})")
print("  rho > 0.3 -> abstention informative | rho ~ 0 -> uncertainty uninformative")
s=one.sort_values("mc_std")
abst=pd.DataFrame([dict(coverage=f"{c*100:.0f}%",n=int(len(s)*c),
    MAE=round(np.abs(s.iloc[:int(len(s)*c)].pred-s.iloc[:int(len(s)*c)].HGB).mean(),3),
    within_tol=f"{100*(np.abs(s.iloc[:int(len(s)*c)].pred-s.iloc[:int(len(s)*c)].HGB)<=CFG['HB_TOL']).mean():.1f}%")
    for c in (.3,.5,.7,.9,1.0)])
print(); print(abst.to_string(index=False))
abst.to_csv(WORK/"results5"/"table7_abstention.csv",index=False)

f,ax=plt.subplots(1,3,figsize=(16,4.5))
ax[0].scatter(one.HGB,one.pred,s=14,alpha=.5,c="steelblue")
lim=[one.HGB.min()-1,one.HGB.max()+1]; ax[0].plot(lim,lim,"k--",lw=1)
ax[0].set_xlabel("measured HGB (g/dL)"); ax[0].set_ylabel("predicted HGB (g/dL)")
ax[0].set_title(f"{BEST}  MAE={np.abs(one.pred-one.HGB).mean():.3f}")
ax[1].scatter(one.mc_std,err,s=14,alpha=.5,c="darkorange")
ax[1].set_xlabel("MC-Dropout uncertainty"); ax[1].set_ylabel("|error| (g/dL)")
ax[1].set_title(f"Spearman rho = {rho:.3f}, p = {pv:.1e}")
ax[2].plot(np.arange(1,len(s)+1)/len(s)*100,np.abs(s.pred-s.HGB).expanding().mean())
ax[2].axhline(BASE_MAE,c="r",ls=":",label="baseline"); ax[2].legend(fontsize=8)
ax[2].set_xlabel("coverage (%)"); ax[2].set_ylabel("MAE (g/dL)"); ax[2].set_title("risk-coverage")
plt.tight_layout(); plt.savefig(WORK/"figures5"/"fig4_regression.png",dpi=160); plt.show()


[output removed: shows per-slide external laboratory data, which is not shared to protect patient privacy]


---
# PART F - Unique-CBC diagnostic (~12 min)


In [15]:
if RUN["UNIQUE_CBC"]:
    uniq=clean.cbc_unique.values
    print(f"patients with a unique CBC record: {uniq.sum()} / {len(clean)}")
    t0=time.time()
    U=pd.concat([train_eval(f,0,True,True,train_subset=uniq,tag="uniq")
                 for f in range(CFG["N_FOLDS"])],ignore_index=True)
    ref=res[(res.exp=="B_frozen_hybrid")&(res.seed==0)]
    a=U.groupby("fold").apply(lambda d: np.abs(d.pred-d.HGB).mean())
    bb=ref.groupby("fold").apply(lambda d: np.abs(d.pred-d.HGB).mean())
    j=pd.concat([a.rename("unique_only"),bb.rename("all_patients")],axis=1).dropna()
    banner("PART F - UNIQUE-CBC TRAINING SET")
    print(j.round(3).to_string())
    print(f"\nmean: unique-only {j.unique_only.mean():.3f} | all {j.all_patients.mean():.3f}"
          f" | diff {j.all_patients.mean()-j.unique_only.mean():+.3f} g/dL")
    print(f"  unique-only trains on ~{U.n_train.mean():.0f} vs ~{ref.n_train.mean():.0f} patients;")
    print("  an improvement DESPITE less data indicates noise in the target.")
    print(f"  [{time.time()-t0:.0f}s]")
    j.to_csv(WORK/"results5"/"table8_unique_cbc.csv")
    for u,nm in ((True,"unique CBC"),(False,"duplicated CBC")):
        d=res[(res.exp==BEST)&(res.cbc_unique==u)]
        if len(d): print(f"  test-time MAE, {nm:16s}: {np.abs(d.pred-d.HGB).mean():.3f} (n={len(d)})")


patients with a unique CBC record: 363 / 917

PART F - UNIQUE-CBC TRAINING SET
      unique_only  all_patients
fold                           
0           2.404         2.126
1           2.616         2.102
2           2.407         2.084
3           2.875         2.249
4           2.507         2.173

mean: unique-only 2.562 | all 2.147 | diff -0.415 g/dL
  unique-only trains on ~290 vs ~734 patients;
  an improvement DESPITE less data indicates noise in the target.
  [464s]
  test-time MAE, unique CBC      : 2.200 (n=1089)
  test-time MAE, duplicated CBC  : 2.109 (n=1662)


---
# PART G - ResNet50 capacity check (~4 min)


In [16]:
if RUN["RESNET50"]:
    t0=time.time()
    r50=train_eval(0,0,True,True,arch="resnet50",tag="r50")
    r18=res[(res.exp=="B_frozen_hybrid")&(res.fold==0)&(res.seed==0)]
    banner("PART G - BACKBONE CAPACITY (fold 0, seed 0, frozen hybrid)")
    print(f"  ResNet18 : MAE {np.abs(r18.pred-r18.HGB).mean():.3f}")
    print(f"  ResNet50 : MAE {np.abs(r50.pred-r50.HGB).mean():.3f}   [{time.time()-t0:.0f}s]")
    print("  Yang et al. 2026 report larger backbones did not improve smear-based prediction.")
    r50.to_csv(WORK/"results5"/"resnet50_fold0.csv",index=False)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 241MB/s]



PART G - BACKBONE CAPACITY (fold 0, seed 0, frozen hybrid)
  ResNet18 : MAE 2.126
  ResNet50 : MAE 2.163   [287s]
  Yang et al. 2026 report larger backbones did not improve smear-based prediction.


---
# PART H - Lab OOD probe + feature-range diagnostic (~5 min)


In [17]:
if RUN["LAB"]:
    lab_ann=pd.read_csv(LABDIR/"labdata_annotations.csv")
    lab_files=sorted(glob.glob(str(LABDIR/"*.png")))

    def med_diam(rgb):
        g=np.asarray(Image.fromarray(rgb).convert("L"))
        try: m=g<threshold_otsu(g)
        except Exception: m=g<g.mean()
        pr=[p.equivalent_diameter for p in regionprops(sklabel(m)) if p.area>50]
        return float(np.median(pr)) if pr else np.nan

    ref_d=np.nanmedian([med_diam(SLIDES[i]) for i in range(30)])
    raw=[np.asarray(Image.open(f).convert("RGB")) for f in lab_files]
    lab_d=np.nanmedian([med_diam(np.asarray(Image.fromarray(r).resize((SP,SP),Image.LANCZOS)))
                        for r in raw])
    SCALE=lab_d/ref_d
    print(f"median RBC diameter @{SP}px: AneRBC {ref_d:.1f} | lab {lab_d:.1f} "
          f"-> scale factor {SCALE:.2f}x")
    LAB_IMG=np.zeros((len(raw),SP,SP,3),np.uint8)
    for i,r in enumerate(raw):
        h,w=r.shape[:2]; ch,cw=int(h/SCALE),int(w/SCALE)
        LAB_IMG[i]=np.asarray(Image.fromarray(r[(h-ch)//2:(h-ch)//2+ch,(w-cw)//2:(w-cw)//2+cw])
                              .resize((SP,SP),Image.LANCZOS))
    LAB_HAND=np.stack([extract_14d(LAB_IMG[i]) for i in range(len(raw))])

    set_seed(0)
    y=clean.HGB.values.astype(np.float32)
    mu_h=HAND8.reshape(-1,14).mean(0); sd_h=HAND8.reshape(-1,14).std(0)+1e-6
    mu_y,sd_y=y.mean(),y.std(); HN=((HAND8-mu_h)/sd_h).astype(np.float32)
    dl=DataLoader(SlideDS(np.arange(len(clean)),(y-mu_y)/sd_y,True,HN),
                  batch_size=CFG["BATCH"],shuffle=True,num_workers=CFG["WORKERS"],drop_last=True)
    final=HybridNet(True,True,CFG["DROPOUT"]).to(DEV)
    opt=torch.optim.AdamW(pgroups(final),weight_decay=CFG["WD"])
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=CFG["EPOCHS_FR"])
    sc=torch.amp.GradScaler("cuda",enabled=(DEV=="cuda"))
    for e in range(CFG["EPOCHS_FR"]):
        final.train()
        for img,hd,yb in dl:
            with torch.amp.autocast("cuda",enabled=(DEV=="cuda")):
                loss=F.huber_loss(final(img.to(DEV),hd.to(DEV)),yb.to(DEV))
            opt.zero_grad(); sc.scale(loss).backward(); sc.step(opt); sc.update()
        sch.step()

    z_lab=(LAB_HAND-mu_h)/sd_h; z_trn=(HAND8[:,0]-mu_h)/sd_h
    banner("FEATURE-RANGE DIAGNOSTIC")
    print(f"  mean |z|, AneRBC training : {np.abs(z_trn).mean():.2f}")
    print(f"  mean |z|, lab slides      : {np.abs(z_lab).mean():.2f}")
    print(f"  max  |z|, lab slides      : {np.abs(z_lab).max():.2f}")
    print("  If lab |z| far exceeds training |z|, the handcrafted stream is")
    print("  extrapolating and alpha_hand must NOT be interpreted.")
    pd.DataFrame({"feature":FEATS,"mean_abs_z_lab":np.abs(z_lab).mean(0).round(2)}
        ).sort_values("mean_abs_z_lab",ascending=False).to_csv(
        WORK/"results5"/"table9_feature_range.csv",index=False)

    def probe(imgs,hands):
        x=torch.tensor(imgs.astype(np.float32)/255.).permute(0,3,1,2).to(DEV)
        h=torch.tensor(((hands-mu_h)/sd_h).astype(np.float32)).to(DEV)
        final.eval()
        with torch.no_grad(): _,a=final(x,h,return_alpha=True)
        for mod in final.modules():
            if isinstance(mod,nn.Dropout): mod.train()
        with torch.no_grad():
            mc=np.stack([final(x,h).cpu().numpy() for _ in range(CFG["MC_PASSES"])])
        lo,hi=CFG["HB_CLAMP"]
        return np.clip(mc.mean(0)*sd_y+mu_y,lo,hi), mc.std(0)*sd_y, a[:,1].cpu().numpy()

    idx=FOLDS[0][1][:60]
    im,isd,ia=probe(np.stack([SLIDES[i] for i in idx]),HAND8[idx,0])
    lm,lsd,la=probe(LAB_IMG,LAB_HAND)
    banner("PART H - OUT-OF-DISTRIBUTION PROBE")
    print(f"  uncertainty in-domain {isd.mean():.4f} | lab {lsd.mean():.4f} "
          f"| ratio {lsd.mean()/isd.mean():.2f}x")
    print(f"  Mann-Whitney (lab > in-domain) p = "
          f"{mannwhitneyu(lsd,isd,alternative='greater')[1]:.2e}")
    print(f"  alpha_hand  in-domain {ia.mean():.3f} | lab {la.mean():.3f}")
    print(f"  lab predictions mean {lm.mean():.2f} sd {lm.std():.2f}  "
          f"(near-zero sd = collapsed to a constant)")
    lab_out=pd.DataFrame(dict(image_id=[Path(f).stem for f in lab_files],
        hb_pred=lm.round(2),uncertainty=lsd.round(3),alpha_hand=la.round(3),
        mean_abs_z=np.abs(z_lab).mean(1).round(2))).merge(
        lab_ann[["image_id","hb_status","morph_verdict","discordant"]],
        on="image_id",how="left")
    print(); print(lab_out.to_string(index=False))
    lab_out.to_csv(WORK/"results5"/"table10_lab.csv",index=False)

    plt.figure(figsize=(6,4))
    bins=np.linspace(0,max(np.percentile(isd,99),np.percentile(lsd,99)),30)
    plt.hist(isd,bins=bins,alpha=.6,density=True,label="AneRBC (in-domain)")
    plt.hist(lsd,bins=bins,alpha=.6,density=True,label="lab slides (OOD)")
    plt.xlabel("MC-Dropout uncertainty (g/dL)"); plt.ylabel("density"); plt.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(WORK/"figures5"/"fig5_ood.png",dpi=160); plt.show()


[output removed: shows per-slide external laboratory data, which is not shared to protect patient privacy]


---
# PART H2 - Lab batch 2: external slides with analyser haemoglobin (~1 min)

18 further slides from the same laboratory, each with a same-day haematology analyser
report matched by sample ID. **Inference only**: the model trained in Part H is applied
unchanged; nothing is fitted or tuned on these slides.

Reference Hb is the printed analyser value (confirmed by the clinician as the run that
corresponds to the photographed smear). WHO cut-offs are sex-specific here because sex is
recorded per patient: 12 g/dL women, 13 g/dL men.

Caveat carried into the paper: the morphology grades were written on the CBC printout,
so the smear read was not blinded to Hb. Morphology-vs-Hb agreement is descriptive only.

In [18]:
H2_DONE=False
B2_CSV = LABDIR/"labdata_batch2_cbc.csv" if LABDIR else None
if RUN["LAB"] and B2_CSV is not None and B2_CSV.exists():
    b2 = pd.read_csv(B2_CSV)
    missing = [f for f in b2.filename if not (LABDIR/f).exists()]
    assert not missing, f"batch-2 images not found next to the CSV: {missing[:5]}"
    raw2 = [np.asarray(Image.open(LABDIR/f).convert("RGB")) for f in b2.filename]

    # same scale matching as Part H, estimated on batch 2 itself
    d2 = np.nanmedian([med_diam(np.asarray(Image.fromarray(r).resize((SP,SP),Image.LANCZOS)))
                       for r in raw2])
    SCALE2 = d2/ref_d
    print(f"batch-2 scale factor {SCALE2:.2f}x (batch 1: {SCALE:.2f}x)")
    IMG2 = np.zeros((len(raw2),SP,SP,3),np.uint8)
    for i,r in enumerate(raw2):
        h,w = r.shape[:2]; ch,cw = int(h/SCALE2),int(w/SCALE2)
        IMG2[i] = np.asarray(Image.fromarray(r[(h-ch)//2:(h-ch)//2+ch,(w-cw)//2:(w-cw)//2+cw])
                             .resize((SP,SP),Image.LANCZOS))
    HAND2 = np.stack([extract_14d(IMG2[i]) for i in range(len(raw2))])

    pm2, psd2, pa2 = probe(IMG2, HAND2)
    b2["hb_pred"]=pm2.round(2); b2["uncertainty"]=psd2.round(3); b2["alpha_hand"]=pa2.round(3)
    b2["mean_abs_z"]=np.abs((HAND2-mu_h)/sd_h).mean(1).round(2)
    err2 = b2.hb_pred-b2.HGB
    H2 = dict(n=len(b2),
              mae=float(err2.abs().mean()),
              bias=float(err2.mean()),
              base_mae=float((b2.HGB-b2.HGB.median()).abs().mean()),
              r=float(pearsonr(b2.hb_pred,b2.HGB)[0]),
              who_acc=float(((b2.hb_pred<b2.who_threshold).astype(int)==b2.who_anemic).mean()),
              morph_agree=float(1-b2.discordant.mean()),
              unc=float(psd2.mean()), unc_in=float(isd.mean()),
              z=float(b2.mean_abs_z.mean()))

    banner(f"PART H2 - EXTERNAL Hb, n = {H2['n']}, one laboratory, inference only")
    print(f"  MAE {H2['mae']:.2f} g/dL | bias {H2['bias']:+.2f} | "
          f"median-predictor MAE on these slides {H2['base_mae']:.2f}")
    print(f"  Pearson r {H2['r']:.2f}   (n = {H2['n']}; descriptive only)")
    print(f"  WHO decision accuracy {H2['who_acc']:.3f} | "
          f"morphology-vs-Hb agreement {H2['morph_agree']:.3f} (not blinded)")
    print(f"  uncertainty {H2['unc']:.3f} vs in-domain {H2['unc_in']:.3f} | "
          f"mean |z| {H2['z']:.2f} (training ~{np.abs(z_trn).mean():.2f})")
    print(f"  prediction range {b2.hb_pred.min():.1f}-{b2.hb_pred.max():.1f} "
          f"vs measured {b2.HGB.min():.1f}-{b2.HGB.max():.1f}")
    print(); print(b2[["image_id","sex","HGB","hb_pred","uncertainty","alpha_hand",
                       "mean_abs_z","morph_text"]].to_string(index=False))
    b2.drop(columns=[c for c in ("prior_episode_date","prior_episode_hb") if c in b2]
            ).to_csv(WORK/"results5"/"table11_lab_batch2.csv", index=False)

    lim=[min(b2.HGB.min(),b2.hb_pred.min())-1, max(b2.HGB.max(),b2.hb_pred.max())+1]
    plt.figure(figsize=(4.5,4.5))
    plt.scatter(b2.HGB,b2.hb_pred,c=b2.morph_abnormal,cmap="coolwarm",edgecolor="k")
    plt.plot(lim,lim,"k--",lw=1); plt.xlim(lim); plt.ylim(lim)
    plt.xlabel("measured HGB (g/dL)"); plt.ylabel("predicted HGB (g/dL)")
    plt.title(f"External batch 2 (n={H2['n']})  MAE {H2['mae']:.2f}")
    plt.tight_layout(); plt.savefig(WORK/"figures5"/"fig6_lab_batch2.png",dpi=160); plt.show()
    H2_DONE=True
else:
    print("PART H2 skipped: labdata_batch2_cbc.csv not found in the lab dataset.")

[output removed: shows per-slide external laboratory data, which is not shared to protect patient privacy]


---
# Summary


In [19]:
banner("V5 SUMMARY - ALL RESULTS")
print("\nPART A - AUDIT")
for k,v in audit.items(): print(f"   {k:32s} {v}")
if RUN["GEOMETRY"]:
    print("\nPART B - GEOMETRY vs ANALYSER (top 6 single predictors)")
    print(geo.head(6).to_string(index=False))
    print("\n   multivariate, 5-fold CV:")
    print(mv[mv.target=="MCV"].to_string(index=False))
print("\nPART C - ABLATION (corrected masks)")
print(tab.drop(columns="_m").to_string(index=False))
print(f"   baseline {BASE_MAE:.3f} g/dL")
print("\n   paired tests:"); print(pairs_tab.to_string(index=False))
print("\nPART D - BINARY DECISION"); print(binary.to_string(index=False))
print(f"\nPART E - ABSTENTION  rho = {rho:.3f} (p = {pv:.1e})")
if RUN["UNIQUE_CBC"]:
    print(f"\nPART F - UNIQUE-CBC  {j.unique_only.mean():.3f} vs {j.all_patients.mean():.3f}")
if RUN["RESNET50"]:
    print(f"\nPART G - R18 {np.abs(r18.pred-r18.HGB).mean():.3f} vs "
          f"R50 {np.abs(r50.pred-r50.HGB).mean():.3f}")
if RUN["LAB"]:
    print(f"\nPART H - scale {SCALE:.2f}x | uncertainty {lsd.mean()/isd.mean():.2f}x | "
          f"lab |z| {np.abs(z_lab).mean():.2f} vs {np.abs(z_trn).mean():.2f}")
if H2_DONE:
    print(f"\nPART H2 - external n={H2['n']} | MAE {H2['mae']:.2f} (baseline {H2['base_mae']:.2f}) | "
          f"bias {H2['bias']:+.2f} | WHO acc {H2['who_acc']:.3f}")
print("\nfiles written:")
for p in sorted(list((WORK/'results5').glob('*'))+list((WORK/'figures5').glob('*'))):
    print("  ",p.relative_to(WORK))
print("\nDownload /kaggle/working/results5 and figures5 before the session ends.")



V5 SUMMARY - ALL RESULTS

PART A - AUDIT
   label_vs_hgb_auc                 0.635
   who_anemic_pct_healthy_folder    50.1
   who_anemic_pct_anemic_folder     72.5
   label_vs_who_agreement_pct       61.4
   label_who_disagreement_pct       38.6
   label_vs_morphology_pct          88.1
   cbc_files                        1000
   cbc_distinct                     641
   cbc_duplicate_pct                35.9
   unusable_records                 82
   clean_patients                   917
   stitch_pixel_diff                0.0

PART B - GEOMETRY vs ANALYSER (top 6 single predictors)
   measured analyser   n  pearson_r  spearman    r2       p
 vol_proxy2      MCH 907      0.195     0.153 0.038 3.3e-09
 vol_proxy2      MCV 907      0.192     0.139 0.037 5.6e-09
area_median      MCV 907      0.174     0.195 0.030 1.3e-07
  diam_mean      MCV 907      0.164     0.128 0.027 7.1e-07
  area_mean      MCV 907      0.162     0.128 0.026 9.2e-07
  vol_proxy      MCV 907      0.160     0.129 0.026 1